## Reading 37-1 - Redundancy and Voting

By the end of this reading, you will learn:
<ol>
    <li>Why flight software must assume that faults <i>will</i> happen</li>
    <li>How triple modular redundancy (TMR) and a majority voter mask a single fault</li>
    <li>The two situations in which voting fails, and why one of them destroyed Ariane 5</li>
    <li>Why the voted value must be written back, and how often</li>
</ol>

### Getting the Reading Code

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your Reading Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir reading37
    cd reading37

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/readings/lec37/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    leaky_bucket.c  Makefile  scrub.c  thermostat.c  tmr_vote.c

### Assume Something Will Break

Every lecture so far has been about <i>preventing</i> faults: rules, standards, and tools that keep mistakes out of the code. But some faults cannot be prevented. In space, a charged particle passing through a memory chip can flip a bit while the program is running. This is called a <b>single-event upset (SEU)</b>, and over a long mission it is not a question of whether one will happen, but how many.

<b>Fault-tolerant</b> software keeps working correctly when something breaks. Amalaye Oyake's talk draws on Robert Hanmer's <i>Patterns for Fault Tolerant Software</i> (2007), which catalogs dozens of proven techniques. Over this lecture we will build five of them in C: <b>Redundancy</b>, <b>Voting</b>, <b>Correcting Audits</b>, <b>Riding Over Transients</b>, and the <b>Leaky Bucket Counter</b>.

### Triple Modular Redundancy - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec37/tmr_vote.c">tmr_vote.c</a>

The simplest way to survive one fault is to keep three copies of everything that matters, and let them vote. This is <b>triple modular redundancy (TMR)</b>. If any two copies agree, their value wins. Ama's slide shows the voter in Ada. Here it is in C:

    typedef struct {
    	int16_t copy[3];
    } tmr_reading;

    int vote(const tmr_reading *r, int16_t *result)
    {
    	if (r->copy[0] == r->copy[1] || r->copy[0] == r->copy[2]) {
    		*result = r->copy[0];
    		return VOTE_OK;
    	}
    	if (r->copy[1] == r->copy[2]) {
    		*result = r->copy[1];
    		return VOTE_OK;
    	}
    	return VOTE_NO_QUORUM;
    }

The example stores a temperature of 21.50 C in hundredths of a degree (2150) and injects bit flips, as you did with the command queue in Lecture 34:

    $ make tmr_vote
    1. No faults                       copies    2150    2150    2150  -> vote    2150
    2. One bit flip, one copy          copies    2150    3174    2150  -> vote    2150
    3. SAME bit flip, two copies       copies    3174    3174    2150  -> vote    3174
    4. DIFFERENT bit flips, two copies copies    3174    2158    2150  -> NO QUORUM: reported, not guessed
    5. Same software bug, all copies   copies  -30536  -30536  -30536  -> vote  -30536

<ul>
    <li><b>Case 2</b> is what TMR is for. Flipping bit 10 added 1024 to one copy, and the voter masked it completely.</li>
    <li><b>Case 4</b> has no majority. The voter <i>reports</i> the problem instead of picking a value. Ada's version of this function refuses to accept an input without a quorum; in C, we return an error code and let the caller decide, as in Reading 35-1.</li>
    <li><b>Case 3</b> is the first limit of voting: two identical wrong copies outvote one right one. As Ama's slide puts it, "two identical wrong copies win."</li>
</ul>

### When All Three Copies Agree and Are All Wrong

Case 5 is the most important line in the output. Three <i>separate</i> sensors all measured 350.000 C during a hot burn. But all three channels ran the <i>same</i> conversion function:

    int16_t to_hundredths(int32_t millidegrees)
    {
    	return (int16_t)(millidegrees / 10);
    }

35000 does not fit in an <code>int16_t</code>, so every channel produced the same wrong value: -305.36 C, the same wraparound you found in <code>hot.txt</code> in Lecture 32. The voter saw three identical values and accepted them unanimously.

This is exactly what happened to Ariane 5 (Reading 32-1). It had two guidance computers, but both ran the same software, so both failed the same way within milliseconds of each other. Ama calls this <b>hidden simplex</b>: hardware that looks redundant, but shares a power supply, a clock, or one software image, so a single fault takes out every copy at once.

Redundancy protects against <b>random</b> faults, like a bit flip in one copy. It cannot protect against <b>systematic</b> faults, like a bug present in every copy. The Boeing 777's primary flight computers attack this directly: each of the three computers contains three channels built on different processor types, with the software built by different compilers, so that a single flaw is unlikely to appear in every channel. And the Power of Ten rules you learned in Lectures 34 and 35 exist precisely because no amount of voting can catch a bug that every copy shares.

Ama's question for every design: <b>what single input can command this actuator?</b>

### Writing Back the Vote - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec37/scrub.c">scrub.c</a>

TMR survives one bad copy. But after a bit flip, that copy <i>stays</i> bad until something fixes it. If a second flip lands in a different copy before then, the vote fails.

The fix is Hanmer's <b>Correcting Audits</b> pattern: after voting, write the voted value back into all three copies. This is called <b>scrubbing</b>. The simulation gives each hour a 1% chance of a bit flip in a random copy, and measures how long the value survives:

    $ make scrub
    Chance of a bit flip each hour: 0.01

    Scrubbing       Mean time to voting failure
    Never                             253 hours
    Once a week                       360 hours
    Once a day                       1479 hours
    Every hour       no failure in 1000 trials of 1000000 hours

Without scrubbing, the value fails in about ten days. Scrubbing once a week barely helps, because a second flip usually arrives within a week of the first. Scrubbing every hour, faster than flips arrive, means a single flip is always erased before another can join it, and in a million simulated hours, about 114 years, the voter never failed.

<b>The lesson:</b> the scrub interval must be shorter than the time between upsets. The same idea is built into spacecraft memory as <b>EDAC</b> (error detection and correction) hardware, which corrects single-bit errors automatically and periodically sweeps through memory to correct them before they accumulate.

### <font color = "red">Class Introduction Question #1 - In <code>tmr_vote.c</code>, why did the voter mask the fault in Case 2 but return the wrong value in Case 3? What does it do in Case 4, and why is that better than picking a value?</font>

### <font color = "red">Class Introduction Question #2 - In Case 5, three separate sensors were used, and the voter still accepted a wrong value. What is hidden simplex, and how did it destroy Ariane 5?</font>

### <font color = "red">Class Introduction Question #3 - Why did scrubbing once a week barely help, while scrubbing every hour prevented every failure in the simulation?</font>

### The next reading for this lecture is <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec37/Reading%2037-2.ipynb">Reading 37-2 - Riding Over Transients</a>